# WMAPE bottom-up — Sección 1 / tienda 00063 / SKU 127360

## Definición correcta (única para todos los niveles)

1. Tomar **solo hojas** `sku+tienda` (`unique_id` con forma `sec||T:tienda||S:sku`).
2. Excluir `period_type == "forecast_only"` y filas con `y == 0` o nulo.
3. En cada fila hoja: `abs_err = |y − ŷ|`.
4. Agregar según el nivel:

| Nivel | Alcance de la suma |
|-------|--------------------|
| SKU+tienda | filas de esa hoja |
| Tienda | filas de **todas** las hojas de esa tienda |
| Sección | filas de **todas** las hojas de la sección |

$$\mathrm{wMAPE} = \frac{\sum \mathrm{abs\_err}}{\sum |y|}$$

**Importante:** tienda y sección **no** usan el `yhat` del nodo agregado (RLS de tienda/sección). Solo errores de hojas.

Referencia previa dashboard: SKU+tienda 29.35%, tienda 8.42%, sección 9.16% (método viejo sobre serie agregada en tienda/sección). Tras bottom-up los % de tienda/sección cambian; el de hoja debe coincidir con Σ|y−ŷ|/Σ|y| de esa serie.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import polars as pl

PROJECT_ROOT = Path.cwd()
for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
    if (candidate / "settings.py").exists() or (candidate / "app" / "backend.py").exists():
        PROJECT_ROOT = candidate
        break
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import settings
from app import backend

FORECAST_PATH = Path(settings.FORECAST_PATH)
print("FORECAST_PATH:", FORECAST_PATH, "exists=", FORECAST_PATH.exists())

In [ ]:
res_df = backend.load_forecast_parquet(FORECAST_PATH)
UNIDAD = "Valor ($)"
has_value = "value" in res_df.columns and "valuehat" in res_df.columns
unit_df = backend.prepare_unit_df(res_df, UNIDAD, has_value)

UID_SECCION = "1"
UID_TIENDA = settings.make_unique_id("1", store="00063")
UID_HOJA = settings.make_unique_id("1", store="00063", sku="127360")
print(UID_HOJA, "|", UID_TIENDA, "|", UID_SECCION)
print("unit_df", unit_df.shape)

## 1. Hojas scorables

In [ ]:
leaves = unit_df.filter(
    pl.col("unique_id").str.count_matches(r"\|\|", literal=False) == 2
)
if "period_type" in leaves.columns:
    leaves = leaves.filter(pl.col("period_type") != "forecast_only")

leaves = leaves.filter(
    pl.col("y").is_not_null() & pl.col("yhat").is_not_null() & (pl.col("y") != 0)
).with_columns(
    (pl.col("y") - pl.col("yhat")).abs().alias("abs_err"),
    pl.col("unique_id").str.replace(r"\|\|S:.*$", "").alias("store_uid"),
    pl.col("unique_id").str.split("||").list.get(0).alias("seccion"),
    pl.col("unique_id").str.extract(r"\|\|S:([^|]+)", 1).alias("sku"),
)
print("Hojas con venta:", leaves.height, "| series:", leaves["unique_id"].n_unique())
leaves.select(["unique_id", "ds", "y", "yhat", "abs_err", "store_uid"]).head(5)

## 2. SKU+tienda

In [ ]:
hoja = leaves.filter(pl.col("unique_id") == UID_HOJA)
sum_y_h = float(hoja["y"].sum()) if hoja.height else 0.0
sum_e_h = float(hoja["abs_err"].sum()) if hoja.height else 0.0
wmape_h = sum_e_h / abs(sum_y_h) if sum_y_h else 0.0
print(f"filas={hoja.height}  Σ|y|={sum_y_h:,.4f}  Σabs_err={sum_e_h:,.4f}")
print(f"wMAPE hoja = {wmape_h*100:.2f}%")

## 3. Tienda bottom-up (todas las hojas de la tienda)

In [ ]:
ti_leaves = leaves.filter(pl.col("store_uid") == UID_TIENDA)
sum_y_t = float(ti_leaves["y"].sum()) if ti_leaves.height else 0.0
sum_e_t = float(ti_leaves["abs_err"].sum()) if ti_leaves.height else 0.0
wmape_t = sum_e_t / abs(sum_y_t) if sum_y_t else 0.0
print(f"n_hojas={ti_leaves['unique_id'].n_unique()}  filas={ti_leaves.height}")
print(f"Σ|y|={sum_y_t:,.4f}  Σabs_err={sum_e_t:,.4f}")
print(f"wMAPE tienda bottom-up = {wmape_t*100:.2f}%")

serie_t = unit_df.filter(pl.col("unique_id") == UID_TIENDA)
if "period_type" in serie_t.columns:
    serie_t = serie_t.filter(pl.col("period_type") != "forecast_only")
serie_t = serie_t.filter(pl.col("y").is_not_null() & (pl.col("y") != 0))
if serie_t.height:
    bad = float((serie_t["y"] - serie_t["yhat"]).abs().sum()) / abs(float(serie_t["y"].sum()))
    print(f"(método viejo serie agregada) = {bad*100:.2f}%  ← incorrecto")

## 4. Sección bottom-up

In [ ]:
sec_leaves = leaves.filter(pl.col("seccion") == UID_SECCION)
sum_y_s = float(sec_leaves["y"].sum()) if sec_leaves.height else 0.0
sum_e_s = float(sec_leaves["abs_err"].sum()) if sec_leaves.height else 0.0
wmape_s = sum_e_s / abs(sum_y_s) if sum_y_s else 0.0
print(f"n_hojas={sec_leaves['unique_id'].n_unique()}  filas={sec_leaves.height}")
print(f"Σ|y|={sum_y_s:,.4f}  Σabs_err={sum_e_s:,.4f}")
print(f"wMAPE sección bottom-up = {wmape_s*100:.2f}%")

serie_s = unit_df.filter(pl.col("unique_id") == UID_SECCION)
if "period_type" in serie_s.columns:
    serie_s = serie_s.filter(pl.col("period_type") != "forecast_only")
serie_s = serie_s.filter(pl.col("y").is_not_null() & (pl.col("y") != 0))
if serie_s.height:
    bad = float((serie_s["y"] - serie_s["yhat"]).abs().sum()) / abs(float(serie_s["y"].sum()))
    print(f"(método viejo serie agregada) = {bad*100:.2f}%  ← incorrecto")

## 5. Resumen + backend.wmape_bottom_up

In [ ]:
manual = pl.DataFrame({
    "unique_id": [UID_HOJA, UID_TIENDA, UID_SECCION],
    "nivel": ["sku+tienda", "tienda bottom-up", "sección bottom-up"],
    "wMAPE_%": [round(wmape_h*100, 2), round(wmape_t*100, 2), round(wmape_s*100, 2)],
    "sum_y": [sum_y_h, sum_y_t, sum_y_s],
    "sum_abs_err": [sum_e_h, sum_e_t, sum_e_s],
})
display(manual)

tabla = backend.wmape_bottom_up(unit_df)
display(
    tabla.filter(pl.col("unique_id").is_in([UID_HOJA, UID_TIENDA, UID_SECCION]))
    .with_columns((pl.col("wmape")*100).round(2).alias("wMAPE_%"))
)

if "period_type" in unit_df.columns:
    tin = backend.wmape_bottom_up(unit_df.filter(pl.col("period_type") == "in_sample"))
    print("Solo in_sample:")
    display(
        tin.filter(pl.col("unique_id").is_in([UID_HOJA, UID_TIENDA, UID_SECCION]))
        .with_columns((pl.col("wmape")*100).round(2).alias("wMAPE_%"))
    )